# Notebook 06 - Mobile labor / reallocation (R5)

Port of `GDP_Simulation_88sectorKLEMS_reallocation.m`.

Key differences from the fixed-labor case (R3):
- **w = 1** — common economy-wide wage (labor is mobile across sectors)
- Only **N price equations** are solved (no quantity equations)
- GDP = CES consumption index: `(β'·p^(1-σ))^(1/(σ-1))`
- Parameters: ε = 0.6, θ = 0.2, σ = 0.9 (from the MATLAB calibration)

In [2]:
using LinearAlgebra, Statistics, Printf, DelimitedFiles

const NB_DIR = @__DIR__
const PKG = joinpath(NB_DIR, "..")
const DATA_DIR = joinpath(PKG, "..", "Replication Files", "GDP Simulatin -- 88 Sector")
const RESULTS_DIR = joinpath(PKG, "data", "results")
mkpath(RESULTS_DIR)
push!(LOAD_PATH, joinpath(PKG, "src"))
include(joinpath(PKG, "src", "BFReplication.jl"))
using .BFReplication
using .BFReplication.DataLoader

println("Module loaded OK")

Module loaded OK


In [3]:
data  = load_bf_data(joinpath(DATA_DIR, "BFdata.csv"); year=1980)
stfp, _, _ = load_tfp_data(joinpath(DATA_DIR, "stfp.csv"))
Sigma_yearly, Sigma_4year = empirical_covariances(stfp)

# Verify: Domar weights
λ = (I - Diagonal(1 .- data.α) * data.Ω)' \ data.β
println("N sectors = $(data.N), sum(λ) = $(round(sum(λ), digits=4))")

N sectors = 76, sum(λ) = 2.0918


## Oil-shock test (sector 7, A = 0.7 / 1.3)

Compare the amplification under mobile labor vs fixed labor.

In [4]:
# Reallocation (mobile labor) solver
ε, θ, σ = 0.6, 0.2, 0.9

for (label, A7) in [("Baseline", 1.0), ("Oil shock (neg)", 0.7), ("Oil shock (pos)", 1.3)]
    A = ones(data.N); A[7] = A7
    sol = solve_bf_realloc(A, data.Ω, data.α, data.β, ε, θ, σ)
    Δ = log(sol.real_gdp)
    λ7 = dot(λ, log.(A))
    amp = Δ / λ7
    println("$label: GDP=$(round(sol.real_gdp, digits=6))  Δlog=$(round(Δ, digits=6))  Hulten=$(round(λ7, digits=6))  amp=$(round(amp, digits=3))x")
end

Baseline: GDP=1.0  Δlog=0.0  Hulten=0.0  amp=Infx
Oil shock (neg): GDP=0.967547  Δlog=-0.032991  Hulten=-0.027681  amp=1.192x
Oil shock (pos): GDP=1.018271  Δlog=0.018106  Hulten=0.020362  amp=0.889x


## Monte Carlo with mobile labor

4-year diagonal covariance (matching the MATLAB reallocation script). Parameters: ε = 0.6, θ = 0.2, σ = 0.9.

In [5]:
TRIALS = 2000
Cov_4yr = Matrix(Diagonal(diag(Sigma_4year)))

mc = run_monte_carlo_realloc(data, Cov_4yr; trials=TRIALS, seed=42)
m = mc.moments

println("Reallocation MC (4-yr cov, $TRIALS trials):")
println("  converged $(mc.n_converged)/$TRIALS; correct $(mc.n_correct)")
println("  mean   = $(round(m.mean*100, digits=4)) %")
println("  std    = $(round(m.std*100, digits=4)) %")
println("  skew   = $(round(m.skewness, digits=3))")
println("  exkurt = $(round(m.excess_kurtosis, digits=3))")

Reallocation MC (4-yr cov, 2000 trials):
  converged 2000/2000; correct 2000
  mean   = -0.9718 %
  std    = 2.4797 %
  skew   = -0.197
  exkurt = 0.034


In [6]:
# Export
open(joinpath(RESULTS_DIR, "mc_loggdp_realloc.csv"), "w") do io
    println(io, "log_gdp")
    for v in mc.log_gdp
        println(io, v)
    end
end
open(joinpath(RESULTS_DIR, "mc_moments_realloc.txt"), "w") do io
    println(io, "mean,std,skewness,excess_kurtosis")
    println(io, "$(m.mean),$(m.std),$(m.skewness),$(m.excess_kurtosis)")
end
println("exported realloc MC results")

exported realloc MC results


## Comparison: Fixed labor vs Mobile labor

The key difference is that mobile labor reduces amplification because labor can reallocate to less-affected sectors.

In [7]:
# Fixed-labor comparison (oil shock)
A70 = ones(data.N); A70[7] = 0.7
sol_fixed = solve_bf(A70, data.Ω, data.α, data.β, data.L, 0.5, 0.001, 0.9)
sol_mobile = solve_bf_realloc(A70, data.Ω, data.α, data.β, ε, θ, σ)

println("Oil shock (A=0.7):")
println("  Fixed labor: Δlog GDP = $(round(log(sol_fixed.C), digits=6))")
println("  Mobile labor: Δlog GDP = $(round(log(sol_mobile.real_gdp), digits=6))")

Oil shock (A=0.7):
  Fixed labor: Δlog GDP = -0.053251
  Mobile labor: Δlog GDP = -0.032991
